# Compute stats results

In [ ]:
import pandas as pd
import numpy as np
import pickle

import folium
import branca.colormap as cm

In [ ]:
# Read the flattened candidates.
path_dict_candidates = './data_simulator/huge_dataset/gencand/dict_flattened_candidates.pkl'
with open(path_dict_candidates, "rb") as f:
    dict_candidates = pickle.load(f)
# display(dict_candidates)

# Read the results of an assessment of fairness based on movement patterns from disk.
path_results = './res_exp.pkl'
with open(path_results, "rb") as f:
    dict_res = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_dict_grids = './data_simulator/huge_dataset/grids/dict_grids.pkl'
with open(path_dict_grids, "rb") as f:
    dict_grids = pickle.load(f)

# Read the geodataframes of the grids (needed to plot the results on a map).
path_unfair_dataset = './synth_labels/unfair_dataset.pkl'
with open(path_unfair_dataset, "rb") as f:
    true_unfair_objs_ids = pickle.load(f)['obj_ids']

For each candidate, retrieve the grid and subset of cell it refers to.

In [ ]:
# Retrieve the set of candidates (subset of cells) that underwent hypothesis testing.
grid_info = dict_candidates['grid_info']
# display(grid_info)

# Compute the number of objects associated with each candidate.
flattened_list_candidates = dict_candidates['flat_ids']
start_pos_candidates = dict_candidates['start_pos']
num_objs_candidates = np.diff(start_pos_candidates)

# Retrieve the log-likelihood ratios computed for the candidates.
extreme_lr_threshold = dict_res['threshold_extreme']
vec_lr_dataset = dict_res['vec_LR_dataset']
vec_inrate_dataset = dict_res['vec_inrate_dataset']
vec_outrate_dataset = dict_res['vec_outrate_dataset']
labels = dict_res['dataset']
# display(vec_lr_dataset)


# For each candidate, here represented as a tuple of cell IDs, associate the grid and subset of cells it refers to.
num_candidates = vec_lr_dataset.size
list_grid_ids = np.empty(num_candidates, dtype=object)
list_cellids = np.empty(num_candidates, dtype=object)
count = 0
for grid in grid_info:
    cell_ids = grid[3].to_numpy()
    num_els_grid = cell_ids.size

    grid_id = np.empty(1, dtype=object)
    grid_id[0] = (int(grid[1]), int(grid[2]))
    grid_id = np.repeat(grid_id, cell_ids.size)    
    
    list_grid_ids[count : count + num_els_grid] = grid_id
    list_cellids[count : count + num_els_grid] = cell_ids

    count += num_els_grid


# Put all the information in a pandas Dataframe.
df_candidates = pd.DataFrame({
    "grid_id":   list_grid_ids,
    "cell_ids":  list_cellids,
    "lr":        vec_lr_dataset.astype(np.float32),
    "num_objs":  num_objs_candidates.astype(np.uint32),
    "in_rate":   vec_inrate_dataset.astype(np.float32),
    "out_rate":  vec_outrate_dataset.astype(np.float32)
})
display(df_candidates)


# Free some memory.
del list_grid_ids, list_cellids, vec_lr_dataset, num_objs_candidates, vec_inrate_dataset, vec_outrate_dataset
del dict_candidates, dict_res

In [ ]:
# Determine which cells have an extreme log-likelihood ratio, according to the simulations' results.
df_candidates_extreme = df_candidates.loc[df_candidates['lr'] >= extreme_lr_threshold].copy()
df_candidates_extreme = df_candidates_extreme.loc[df_candidates_extreme['grid_id'] >= (200, 0)]
df_candidates_extreme.reset_index(names='candidate_id', inplace = True)
display(df_candidates_extreme)

In [ ]:
start_obj_candidates = start_pos_candidates[df_candidates_extreme['candidate_id']]
end_obj_candidates = start_pos_candidates[df_candidates_extreme['candidate_id'] + 1]
# display(start_obj_candidates, end_obj_candidates)

# Find out the IDs of the objects that appear in at least an "extreme" cell.
lists_objs = np.concatenate([flattened_list_candidates[start_obj_candidates[i] : end_obj_candidates[i]] for i in range(len(start_obj_candidates))])
ids_objects_involved = np.unique(lists_objs)
ids_objects_involved.size

In [ ]:
# Determine the intersection between the objects associated with areas truly treated unfairly vs the objects associated 
# with 'extreme' candidates.
intersect_obj_ids = np.intersect1d(true_unfair_objs_ids, ids_objects_involved)
print(ids_objects_involved.size, intersect_obj_ids.size, true_unfair_objs_ids.size)
print(f"Sensitivity: {intersect_obj_ids.size / true_unfair_objs_ids.size}, PPV {intersect_obj_ids.size / ids_objects_involved.size}")